# The neutrino-mass ladder

Data, sources, and status live in [`anomaly.yaml`](anomaly.yaml) (loaded below, never
re-typed as literals in this notebook). Decisions made at each step, including the choices
behind the step-6 fit, are logged in [`decisions.md`](decisions.md).

This notebook ships already solved so that `pytest --nbmake` can confirm it runs top to bottom
in CI. To work through it as an exercise yourself, make your own copy first and avoid reading
`checkpoints_def.py` / `solutions/` until you want a hint or to check your answer.

In [1]:
import sys
from pathlib import Path

import sympy as sp
from IPython.display import Math, display

# nbclient/nbmake set the kernel's cwd to this notebook's own directory; make sure it (and the
# repo root two levels up, for the `anomalies`/`checkpoints`/`fit`/`feynlag_anomalies` packages)
# are importable regardless of how the kernel was launched.
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR.parents[1]
for p in (NB_DIR, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from feynlag_anomalies.registry import load as load_anomaly
from anomalies.neutrino_mass import checkpoints_def as cd

anomaly = load_anomaly("neutrino_mass")
print(anomaly.title, "--", anomaly.maturity)

Neutrino mass and oscillations -- A3


## Step 0 -- back-of-envelope estimate

Dimensional analysis: if a neutrino mass comes from a Yukawa coupling `y`, the electroweak scale
`v`, and gets suppressed by a heavy scale `M`, what combination has units of mass?

$$m_\nu \sim \frac{y^2 v^2}{M}$$

Try it with an $O(1)$ Yukawa, $v = 246\,\text{GeV}$, and a GUT-ish $M \sim 10^{14}\,\text{GeV}$.

In [2]:
from anomalies.neutrino_mass.solutions.step_0 import m_nu_estimate

answer_0 = m_nu_estimate(cd._P0_Y, cd._P0_V, cd._P0_M)
cd.check_0(answer_0)

[step_0] correct -- got 6.0516e-10


True

## Step 1 -- try it with the SM

Enumerate every gauge-invariant Yukawa/mass term at dimension <= 4 for the *minimal* SM lepton
content: a lepton doublet `Ll`, a charged-lepton singlet `eR`, and the Higgs doublet `H` --
**no** right-handed neutrino. `feynlag.suggest.suggest_yukawa` is the oracle: it enumerates every
invariant contraction and re-verifies each one (gauge, discrete, hermiticity, mass-dimension).

In [3]:
from feynlag import suggest_yukawa

from anomalies.neutrino_mass.solutions.step_1 import build_sm_lepton_fields

Ll, eR, H, groups = build_sm_lepton_fields()
terms_dim4 = suggest_yukawa([Ll, eR], [H], list(groups), max_dim=4)
for t in terms_dim4:
    display(Math(rf"\text{{{t.label} (dim {t.dim})}}:\quad {sp.latex(t.expr)}"))

answer_1 = len(terms_dim4)
cd.check_1(answer_1)

<IPython.core.display.Math object>

[step_1] correct -- got 1


True

Only the charged-lepton Yukawa survives: **no dimension<=4 neutrino-mass term exists** for this
field content. The protection blocking it is the combination of `field_content` (no $\nu_R$) and
`accidental_symmetry` (the SM's accidental lepton number) -- see
[`docs/protections.md`](../../docs/protections.md).

## Step 2 -- EFT before a model

Raise `max_dim` to 5. A new invariant should appear: the dimension-5 Weinberg operator
$(LH)(LH)/\Lambda$, a same-chirality (Majorana) contraction of the lepton doublet with two Higgs
doublets.

In [4]:
terms_dim5 = suggest_yukawa([Ll, eR], [H], list(groups), max_dim=5)
for t in terms_dim5:
    display(Math(rf"\text{{{t.label} (dim {t.dim})}}:\quad {sp.latex(t.expr)}"))

answer_2 = any("Weinberg" in t.label for t in terms_dim5)
cd.check_2(answer_2)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

[step_2] correct -- got True


True

With $m_\nu \sim v^2/\Lambda$ (an $O(1)$ Wilson coefficient), what scale $\Lambda$ does the
Weinberg operator need?

For a data-driven $m_\nu$, use the atmospheric splitting from the loaded `Anomaly` (NuFIT 6.0,
normal ordering). The heaviest light neutrino has $m_3 \geq \sqrt{\Delta m^2_{31}}$, with
equality when the lightest state is massless, so $\sqrt{\Delta m^2_{31}}$ is a natural
benchmark. The step-0 illustrative $m_\nu$ is printed alongside it for comparison.

In [5]:
import math

from anomalies.neutrino_mass.solutions.step_2 import lambda_estimate

dm2_31 = {o.name: o for o in anomaly.observables}["Delta m^2_31 (atmospheric, normal ordering)"]
m_nu_atm = math.sqrt(dm2_31.value) * 1e-9  # eV -> GeV

print(f"step-0 illustrative: m_nu ~ {answer_0:.3g} GeV -> Lambda ~ {lambda_estimate(answer_0, cd._P0_V):.3g} GeV")
Lambda_estimate = lambda_estimate(m_nu_atm, cd._P0_V)
print(f"sqrt(Delta m^2_31):  m_nu ~ {m_nu_atm:.3g} GeV -> Lambda ~ {Lambda_estimate:.3g} GeV   (computed)")

step-0 illustrative: m_nu ~ 6.05e-10 GeV -> Lambda ~ 1e+14 GeV
sqrt(Delta m^2_31):  m_nu ~ 5.01e-11 GeV -> Lambda ~ 1.21e+15 GeV   (computed)


## Step 3 -- from the operator to the fields

Which tree-level UV completion is minimal? Following de Blas, Criado, Perez-Victoria, Santiago
(arXiv:1711.10391): fewer fields, smaller representations, fewer parameters, no ad hoc
symmetries wins. Adding one gauge-singlet fermion $\nu_R$ (type-I seesaw) beats a scalar triplet
(type-II) or a fermion triplet (type-III) on every count. This repo never declares that
Lagrangian itself -- it imports the already-verified model from `feynlag-models` by `model_id`.

In [6]:
from feynlag_models.registry import build, metadata

answer_3 = "seesaw_type1"
cd.check_3(answer_3)

print("feynlag-models maturity:", metadata(answer_3)["maturity_level"])
bundle = build(answer_3)
print("bundle.extra keys:", sorted(bundle.extra.keys()))

[step_3] correct -- got 'seesaw_type1'
feynlag-models maturity: 2


bundle.extra keys: ['CPL', 'D', 'LMaj', 'LYukD', 'MH', 'MN1', 'MN2', 'MR', 'MRmat', 'MW', 'MZ', 'Mnu', 'U', 'chi', 'ew', 'heavy', 'light', 'mD', 'mDsym', 'm_light_approx', 'masses', 'nuR', 'rot', 'ufo', 'yv']


## Step 4 -- predict before running

Before reading off any numbers: with exactly one $\nu_R$ (one generation), how many physical
Majorana mass eigenstates should the seesaw mechanism produce? (Hint: think about the rank of
the $2\times2$ matrix $\begin{pmatrix}0 & m_D \\ m_D & M_R\end{pmatrix}$.)

In [7]:
answer_4 = len(bundle.extra["masses"])
check_4 = cd.make_check_4(bundle)
check_4(answer_4)

[step_4] correct -- got 2


True

In [8]:
display(Math(r"\mathcal{M}_\nu = " + sp.latex(bundle.extra["Mnu"])))
display(Math(r"m_\nu^{\rm light} \simeq " + sp.latex(bundle.extra["m_light_approx"])))

values = bundle.values()
light_mass = values[bundle.extra["MN1"].s]
heavy_mass = values[bundle.extra["MN2"].s]
print(f"light Majorana mass:  {light_mass:.3e} GeV  (benchmark: yv=1e-6, M_R=1000 GeV -> ~0.03 eV)")
print(f"heavy Majorana mass:  {heavy_mass:.3e} GeV")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

light Majorana mass:  3.031e-11 GeV  (benchmark: yv=1e-6, M_R=1000 GeV -> ~0.03 eV)
heavy Majorana mass:  1.000e+03 GeV


## Step 5 -- break it on purpose

A single $\nu_R$ gives one light mass -- but oscillation experiments measure **two**
independent $\Delta m^2$ (solar and atmospheric). A single light mass cannot reproduce two
independent splittings: the light-neutrino mass matrix from one $\nu_R$ has rank 1, and two
independent $\Delta m^2$ require rank >= 2, i.e. at least two $\nu_R$.

That gap is written up as a model request rather than built here (this repo never declares a
Lagrangian, and no change is made to `feynlag-models` without approval): see
[`model_requests/seesaw_type1_nN.md`](../../model_requests/seesaw_type1_nN.md).

In [9]:
request_path = Path("../../model_requests/seesaw_type1_nN.md")
answer_5 = request_path.exists()
print(request_path.read_text()[:400] + "...")
cd.check_5(answer_5)

# Model request: `seesaw_type1_nN` (n-generation type-I seesaw)

**Requested by**: `anomalies/neutrino_mass` (step 5 of `ladder.ipynb`)
**Status**: fulfilled on 2026-09-25 by `feynlag-models`' `seesaw_type1_2n` (PR #9, maturity L2).
The id is lowercase because the metadata schema requires `^[a-z0-9_]+$`. It has three lepton
generations (the rank argument needs n_L ≥ 2, and `sm.pieces` offers only ...
[step_5] correct -- got True


True

## Step 6 -- confront with data: a stage-1 $\chi^2$

The two-$\nu_R$ model the step-5 request asked for now exists in `feynlag-models` as
`seesaw_type1_2n` (three lepton generations, two $\nu_R$, maturity L2). Its light-neutrino
matrix has rank 2, so it predicts one **massless** neutrino and two massive ones.

We fit its six Dirac Yukawas $y^\nu_{a b}$ (with $M_1 = 1$ TeV and $M_2 = 3$ TeV held at the
model benchmark) to five measured observables in normal ordering: $\Delta m^2_{21}$,
$\Delta m^2_{31}$, $\theta_{12}$, $\theta_{13}$ and $\theta_{23}$. The values and their source
(NuFIT 6.0) come from the loaded `Anomaly`. $\delta_{CP}$ is recorded but not fitted, because the
model's Yukawas are real, so it cannot produce CP violation.

Each trial point substitutes the Yukawas into the model's symbolic $5\times5$ mass matrix and
diagonalises it with feynlag's numeric Takagi factorisation. There is no model rebuild per point.

**Before running:** six parameters against five observables leaves $-1$ degrees of freedom.
What does a $\chi^2_{\min} \approx 0$ tell you, and what does it *not* tell you?


In [10]:
from anomalies.neutrino_mass.solutions import step_6

model_id = step_6.MODEL_ID
print("feynlag-models maturity:", metadata(model_id)["maturity_level"])
bundle_2n = build(model_id)
display(Math(r"\mathcal{M}_\nu = " + sp.latex(bundle_2n.extra["Mnu"])))
observed = step_6.observed_from_anomaly(anomaly)
print("source:", anomaly.sources[0].identifier, "| consulted", anomaly.sources[0].consulted_on)

fit = step_6.run_fit(bundle_2n, observed)
print(f"\n{'observable':45s} {'observed':>22s} {'fitted':>12s} {'pull':>9s}")
for name, (value, sigma) in observed.items():
    print(f"{name:45s} {value:12.5g} ± {sigma:<8.3g} {fit['predicted'][name]:12.5g} {fit['pulls'][name]:+9.1e}")
print(f"\nchi2_min = {fit['chi2']:.2e}, ndof = {fit['ndof']}, converged: {fit['success']}, "
      f"perturbative: {fit['perturbative']}")

feynlag-models maturity: 2


<IPython.core.display.Math object>

source: arXiv:2410.05380v2 (NuFIT 6.0; JHEP 12 (2024) 216; INSPIRE 2838825), Table 1, IC24 with SK atmospheric data, Normal Ordering | consulted 2026-09-25



observable                                                  observed       fitted      pull
Delta m^2_21 (solar)                              7.49e-05 ± 1.9e-06      7.49e-05  -2.9e-09
Delta m^2_31 (atmospheric, normal ordering)       0.002513 ± 2e-05        0.002513  -2.1e-10
theta_12                                             33.68 ± 0.715           33.68  +6.4e-09
theta_13                                              8.56 ± 0.11             8.56  -6.8e-09
theta_23                                              43.3 ± 0.9              43.3  -1.6e-07

chi2_min = 2.65e-14, ndof = -1, converged: True, perturbative: True


A $\chi^2_{\min}$ of essentially zero with negative degrees of freedom means the model **can**
accommodate the data; it is not evidence that the data **prefer** it. The model's genuine
prediction is structural: the lightest neutrino is exactly massless, so
$\sum m_\nu = \sqrt{\Delta m^2_{21}} + \sqrt{\Delta m^2_{31}}$ is fixed by the splittings alone.
Confronting that sum with cosmology, and the model's effective Majorana mass with neutrinoless
double beta decay, are the natural next constraints; they are not done here.

For contrast, the SM's massless neutrinos predict both splittings to be zero.


In [11]:
m1, m2, m3 = fit["masses_eV"]
print(f"light masses [eV]: m1 = {m1}, m2 = {m2:.4g}, m3 = {m3:.4g}   (computed)")
print(f"sum m_nu = {fit['sum_m_nu_eV']:.4g} eV   (computed; lightest state massless)")
print(f"chi2 over the two splittings: SM = {fit['chi2_sm_dm2']:.3g}, model = {fit['chi2_model_dm2']:.1e}")
print("best-fit Yukawas:", {k: f"{v:.3e}" for k, v in fit["yv"].items()})

answer_6 = fit["predicted"][step_6.FIT_OBSERVABLES[1]] / fit["predicted"][step_6.FIT_OBSERVABLES[0]]
cd.make_check_6(anomaly)(answer_6)


light masses [eV]: m1 = 0.0, m2 = 0.008654, m3 = 0.05013   (computed)
sum m_nu = 0.05878 eV   (computed; lightest state massless)
chi2 over the two splittings: SM = 1.73e+04, model = 8.7e-18
best-fit Yukawas: {'yv_e1': '1.595e-07', 'yv_e2': '5.396e-07', 'yv_mu1': '9.046e-07', 'yv_mu2': '-4.503e-07', 'yv_tau1': '8.916e-07', 'yv_tau2': '6.389e-07'}
[step_6] correct -- got 33.5514


True